# Merge biological replicates of image batches

In [ ]:
import os
import re
import shutil
import numpy as np

# Set up directory paths
dir_1 = "../model_inputs/2024_01_23_Batch_7/"
dir_2 = "../model_inputs/2024_02_06_Batch_8/"
merged_dir = "../model_inputs/2024_01_Batch_7-8_merged/"

# Ensure merged_dir exists
os.makedirs(merged_dir, exist_ok=True)

# Helper function to get allele file mapping for a directory.
def get_allele_files(dir_path):
    """
    Returns a dict mapping allele -> {file_type: full_path}.
    Expected file pattern: {allele}_{file_type}.npy, where file_type is label, pro, or nuc.
    """
    allele_files = {}
    pattern = re.compile(r'^(?P<allele>.+)_(?P<type>label|pro|nuc)\.npy$')
    for filename in os.listdir(dir_path):
        if filename.endswith('.npy'):
            match = pattern.match(filename)
            if match:
                allele = match.group("allele")
                file_type = match.group("type")
                allele_files.setdefault(allele, {})[file_type] = os.path.join(dir_path, filename)
            else:
                print(f"Warning: {filename} in {dir_path} does not match expected pattern")
    return allele_files


# Get file mappings from both directories.
allele_files1 = get_allele_files(dir_1)
allele_files2 = get_allele_files(dir_2)

# Determine the union of alleles across both directories.
alleles = set(allele_files1.keys()) | set(allele_files2.keys())

# Expected file types for each allele
expected_types = {"label", "pro", "nuc"}

for allele in sorted(alleles):
    # Check for missing files in each directory that contains the allele.
    if allele in allele_files1:
        files1 = allele_files1[allele]
        missing = expected_types - files1.keys()
        if missing:
            raise Exception(f"Allele {allele} in {dir_1} is missing files: {missing}")
    if allele in allele_files2:
        files2 = allele_files2[allele]
        missing = expected_types - files2.keys()
        if missing:
            raise Exception(f"Allele {allele} in {dir_2} is missing files: {missing}")
    
    # If allele exists in both directories, stack the arrays.
    if allele in allele_files1 and allele in allele_files2:
        for file_type in expected_types:
            path1 = allele_files1[allele][file_type]
            path2 = allele_files2[allele][file_type]
            arr1 = np.load(path1, allow_pickle=True)
            arr2 = np.load(path2, allow_pickle=True)
            merged_arr = np.concatenate([arr1, arr2], axis=0)
            out_filename = f"{allele}_{file_type}.npy"
            out_path = os.path.join(merged_dir, out_filename)
            np.save(out_path, merged_arr)
            print(f"Stacked and saved {out_filename}")
    # If allele exists only in one directory, copy the files over.
    else:
        src_dir = dir_1 if allele in allele_files1 else dir_2
        allele_files = allele_files1.get(allele, allele_files2.get(allele))
        for file_type in expected_types:
            src_path = allele_files[file_type]
            dest_path = os.path.join(merged_dir, os.path.basename(src_path))
            shutil.copy(src_path, dest_path)
            print(f"Copied {os.path.basename(src_path)} from {src_dir} to merged")

Stacked and saved ABL1_label.npy
Stacked and saved ABL1_nuc.npy
Stacked and saved ABL1_pro.npy
Copied ACSF3_label.npy from ../model_inputs/2024_01_23_Batch_7/ to merged
Copied ACSF3_nuc.npy from ../model_inputs/2024_01_23_Batch_7/ to merged
Copied ACSF3_pro.npy from ../model_inputs/2024_01_23_Batch_7/ to merged
Copied ACSF3-Ala197Thr_label.npy from ../model_inputs/2024_01_23_Batch_7/ to merged
Copied ACSF3-Ala197Thr_nuc.npy from ../model_inputs/2024_01_23_Batch_7/ to merged
Copied ACSF3-Ala197Thr_pro.npy from ../model_inputs/2024_01_23_Batch_7/ to merged
Stacked and saved ACSF3-Arg10Trp_label.npy
Stacked and saved ACSF3-Arg10Trp_nuc.npy
Stacked and saved ACSF3-Arg10Trp_pro.npy
Copied ACSF3-Arg471Trp_label.npy from ../model_inputs/2024_01_23_Batch_7/ to merged
Copied ACSF3-Arg471Trp_nuc.npy from ../model_inputs/2024_01_23_Batch_7/ to merged
Copied ACSF3-Arg471Trp_pro.npy from ../model_inputs/2024_01_23_Batch_7/ to merged
Copied ACSF3-Arg558Trp_label.npy from ../model_inputs/2024_01_23_B